In [ ]:
!pip install -U -q openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.1 MB/s eta 0:00:00


In [1]:
import os
import openai
# Set the API key
with open("../OPENAI_API_Key.txt", "r") as f:
  openai.api_key = ' '.join(f.readlines())

FileNotFoundError: [Errno 2] No such file or directory: '../OPENAI_API_Key.txt'

# Chat Completion Parameters

You are already familiar with a few of the parameters such as `temperature`, `top_p`,`max_tokens`,`stop`, `frequency_penalty`,`presence_penalty` etc.
<br>
ChatCompletions API offers additional capabilities as described in its [API documentation](https://platform.openai.com/docs/api-reference/chat/create)

In this notebook, we will explore a few additional parameters for the ChatCompletions API such as:
  * `logprobs`
  * `top_logprobs`
  * `seed`




## Seed Parameter

The `seed` parameter allows you to acheive more consistent outputs. It helps you control the deterministic nature of the answers. In other words, making the same requests with the same value of `seed` parameter will return the same results, ensuring reproducibility.

You can check the `system_fingerprint` value in the output to see the current  combination of model weights, infrastructure, configuration etc used by the servers.  The `seed` is stored with the prompt in order to replicate the response.



**Value**: By setting the value of the `seed` parameter as an integer (e.g, 123), you can control the determinism in outputs.

**Consistency in behaviour**: If you set the `seed` parameter to the same value across requests, while also keeping other parameters like `prompt`, `temperature`, and `top_p` consistent, the model likely produce identical outputs.




Please note that the `seed` feature is in beta phase and is currently supported only for `gpt-4-1106-preview` and `gpt-3.5-turbo-1106` models.

In [ ]:
def chat_response_nonseeded(system_message, user_request):
  messages = [
      {"role": "system", "content": system_message},
      {"role": "user", "content": user_request},
    ]
  response = openai.chat.completions.create(
      model = "gpt-3.5-turbo-1106",
      temperature = 0.5,
      messages=messages,
      max_tokens=200
    )
  return response.choices[0].message.content

In [ ]:
def chat_response_seeded(system_message, user_request):
  messages = [
      {"role": "system", "content": system_message},
      {"role": "user", "content": user_request},
    ]
  chat_response = openai.chat.completions.create(
      model = "gpt-3.5-turbo-1106",
      temperature = 0.7,
      messages=messages,
      seed=234,
      max_tokens=200,
    )
  return chat_response.choices[0].message.content

In [ ]:
import time
system_message = "You are a chatbot that the user can discuss with. Answer user questions in a conversational way."
user_request = "Output a fruit"
non_seed_answer = []
seed_answer = []
i = 0
while (i < 5):
  print(f"Loop {i}")
  ans1 = chat_response_nonseeded(system_message, user_request)
  non_seed_answer.append(ans1)
  print("Non Seeded Answer:",non_seed_answer[i])
  ans2 = chat_response_seeded(system_message, user_request)
  seed_answer.append(ans2)
  print("Seeded Answer:",seed_answer[i])
  i+=1
  print('='*10)
  time.sleep(10)

Loop 0
Non Seeded Answer: How about an apple? It's a classic choice and comes in many delicious varieties!
Seeded Answer: How about a juicy and delicious apple?
Loop 1
Non Seeded Answer: How about a juicy, ripe mango?
Seeded Answer: How about a juicy and delicious apple?
Loop 2
Non Seeded Answer: How about a juicy, ripe mango?
Seeded Answer: How about a juicy and delicious apple?
Loop 3
Non Seeded Answer: How about a juicy, ripe peach?
Seeded Answer: How about a juicy and delicious apple?
Loop 4
Non Seeded Answer: Sure, how about a juicy and delicious apple?
Seeded Answer: How about a juicy and delicious apple?


An important thing to understand is the difference between `seed` and `temperature` parameters.
<br>

*   While seed is used to control the *randomness* or *stochasticity* of the outputs, `temperature` is used to control the *creativity* or *variability* of generated text.
*   `seed` value ensures that there is a consistency in the outputs generated by requests with the same parameters, and `temperature` focusses more on randomness in selection of next tokens.
* `seed` is assigned an integer value, while `temperature` is assigned float values between 0 and 2
<br>

However, you should remember that determinism cannot be always guaranteed due to the inherent non determinism in LLM models, but this approach will lead to somewhat consistent results. <br>

## Logprobs Parameter

When logprobs is enabled, the API returns the log probabilities of each output token, along with a limited number of the most likely tokens at each token position and their log probabilities. The relevant request parameters are:
- `logprobs`: Whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message. This option is currently not available on the gpt-4-vision-preview model.
- `top_logprobs`: An integer between 0 and 5 specifying the number of most likely tokens to return at each token position, each with an associated log probability. logprobs must be set to true if this parameter is used.
Log probabilities of output tokens indicate the likelihood of each token occurring in the sequence given the context. To simplify, a logprob is log(p), where p = probability of a token occurring at a specific position based on the previous tokens in the context. Some key points about logprobs:

Higher log probabilities suggest a higher likelihood of the token in that context. This allows users to gauge the model's confidence in its output or explore alternative responses the model considered.
Logprob can be any negative number or 0.0. 0.0 corresponds to 100% probability.

Logprobs allow us to compute the joint probability of a sequence as the sum of the logprobs of the individual tokens. This is useful for scoring and ranking model outputs. Another common approach is to take the average per-token logprob of a sentence to choose the best generation.
We can examine the logprobs assigned to different candidate tokens to understand what options the model considered plausible or implausible.


In [ ]:
from math import exp
import pandas as pd
import numpy as np
import os

In [ ]:
def chat_response(system_message, user_request):
  messages = [
      {"role": "system", "content": system_message},
      {"role": "user", "content": user_request},
    ]
  response = openai.chat.completions.create(
      model = "gpt-3.5-turbo-1106",
      temperature = 0.5,
      messages=messages,
      max_tokens=200,
      logprobs=True,
      top_logprobs=2
    )
  return response

SYS_MSG = 'You are a helpful AI Assistant'
USR_RQST = 'What is the capital of France?'
response = chat_response(SYS_MSG, USR_RQST)
output_reponse = response.choices[0].message.content
top_two_logprobs = response.choices[0].logprobs.content[0].top_logprobs
print(output_reponse, top_two_logprobs)

The capital of France is Paris. [TopLogprob(token='The', bytes=[84, 104, 101], logprob=-0.0002291655), TopLogprob(token='Paris', bytes=[80, 97, 114, 105, 115], logprob=-8.99922)]


In [ ]:
response #Detailed ChatCompletion Object

ChatCompletion(id='chatcmpl-95D9lnPph8N5H1Pm5gjt5ITH5NLl3', choices=[Choice(finish_reason='stop', index=0, logprobs=ChoiceLogprobs(content=[ChatCompletionTokenLogprob(token='The', bytes=[84, 104, 101], logprob=-0.0002291655, top_logprobs=[TopLogprob(token='The', bytes=[84, 104, 101], logprob=-0.0002291655), TopLogprob(token='Paris', bytes=[80, 97, 114, 105, 115], logprob=-8.99922)]), ChatCompletionTokenLogprob(token=' capital', bytes=[32, 99, 97, 112, 105, 116, 97, 108], logprob=-3.0545007e-06, top_logprobs=[TopLogprob(token=' capital', bytes=[32, 99, 97, 112, 105, 116, 97, 108], logprob=-3.0545007e-06), TopLogprob(token=' ', bytes=[32], logprob=-13.492818)]), ChatCompletionTokenLogprob(token=' of', bytes=[32, 111, 102], logprob=-0.00036822853, top_logprobs=[TopLogprob(token=' of', bytes=[32, 111, 102], logprob=-0.00036822853), TopLogprob(token=' city', bytes=[32, 99, 105, 116, 121], logprob=-7.908575)]), ChatCompletionTokenLogprob(token=' France', bytes=[32, 70, 114, 97, 110, 99, 101]

Example 2

In [ ]:
LAPTOP_PROD_DESC = """You will be given the laptop specifications and product description for a laptop.
Based on the description, classify the laptop into one of the following categories: general, business, student, gamer, content, programmer, multimedia.
Return only the name of the category, and nothing else.
MAKE SURE your output is one of the seven categories stated.
Laptop Description: {description}"""

In [ ]:
description_1 = """
The HP EliteBook is a premium laptop designed for professionals who value performance and security.
It features an Intel Core i7 processor clocked at 2.8 GHz, providing exceptional speed and power for demanding tasks.
With 16GB of RAM and an SSD, it offers fast and efficient multitasking and storage capabilities.
The laptop sports a 14" LED display with a resolution of 1920x1080, delivering sharp visuals and vibrant colors.
It also comes with an Intel UHD GPU for decent graphical performance.
Weighing just 1.5 kg, it is lightweight and highly portable.
The laptop is equipped with a fingerprint sensor for secure and convenient login.
With a three-year warranty and a battery life of up to 8 hours, it ensures long-lasting reliability.
Priced at 90,000, the HP EliteBook provides top-notch features and durability for professional users.
"""

In [ ]:
SYSTEM_MSG = "You are a helpful AI assistant that can evaluate laptop products based on their description."
USR_RQST = LAPTOP_PROD_DESC.format(description=description_1)
response = chat_response(SYS_MSG, USR_RQST)
response.choices[0].message.content

'business'

In [ ]:
response.choices[0].logprobs.content[0].top_logprobs

[TopLogprob(token='business', bytes=[98, 117, 115, 105, 110, 101, 115, 115], logprob=-0.044131428),
 TopLogprob(token='Business', bytes=[66, 117, 115, 105, 110, 101, 115, 115], logprob=-3.1586373)]